## **Voting Ensemble Regressor**

### **Topic Roadmap**

**1. Load a regression dataset**

**2. Compare base regressors**

**3. Combine predictions with weighted voting**

**4. Evaluate the ensemble**

**5. Key revision notes**

## **1. Dataset and Split**

The original notebook used the removed Boston loader. The maintained Diabetes dataset provides the same regression-ensemble learning workflow.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import load_diabetes

In [2]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## **2. Base Regressors**

The ensemble combines a linear model, a tree, and a support-vector regressor. Scaling is required for the SVR, so it is wrapped in a pipeline.

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

estimators = [
    ("linear", LinearRegression()),
    ("tree", DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)),
    ("svr", make_pipeline(StandardScaler(), SVR(C=10, epsilon=0.1))),
]
for name, model in estimators:
    model.fit(X_train, y_train)
    print(name, f"R2 = {model.score(X_test, y_test):.3f}")

linear R2 = 0.453
tree R2 = 0.334
svr R2 = 0.494


## **3. Voting Regressor**

A voting regressor averages the continuous predictions of its fitted base regressors. Weights can emphasize stronger or more reliable models.

In [4]:
from sklearn.ensemble import VotingRegressor

voter = VotingRegressor(estimators=estimators, weights=[2, 1, 2])
voter.fit(X_train, y_train)
predictions = voter.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, predictions):.2f}")
print(f"RMSE: {mean_squared_error(y_test, predictions) ** 0.5:.2f}")
print(f"R2: {r2_score(y_test, predictions):.3f}")

MAE: 41.28
RMSE: 51.44
R2: 0.501


### **Key Revision Notes**

- Voting regression averages predictions rather than class labels.
- A weighted average can improve performance when model quality differs.
- Preprocessing must be inside a pipeline to avoid leakage.
- The ensemble cannot correct consistently shared errors among all base models.